# BERT

In [25]:
import random
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, confusion_matrix, roc_curve, auc
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer, BertConfig, Trainer, TrainingArguments, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import torch

In [3]:
MODEL_NAME = 'DeepPavlov/rubert-base-cased'
BATCH_SIZE = 16
MAX_LENGTH = 512
NUM_EPOCHS = 5
RANDOM_SEED = 42

In [4]:
def seed_all(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_all(RANDOM_SEED)

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# text = "Всем привет я Дмитрий! Добро пожаловать на лекцию по NLP"
# encoding = tokenizer(
#     text,
#     truncation=True,
#     padding='max_length',
#     max_length=MAX_LENGTH,
#     return_tensors='pt'
# )
# print(encoding)
# from termcolor import colored

# tokens = tokenizer.convert_ids_to_tokens(encoding['input_ids'][0])
# attention_mask = encoding['attention_mask'][0]
# input_ids = encoding['input_ids'][0].tolist()

# # Собираем только полезные токены (без PAD)
# real_tokens = [t for t, m in zip(tokens, attention_mask) if m == 1]
# real_ids = [id_ for id_, m in zip(input_ids, attention_mask) if m == 1]

# # Цвета для чередования
# colors = ['red', 'green', 'yellow', 'blue', 'magenta', 'cyan', 'white']

# # Заголовок с номерами токенов
# print("\n№ токена:  ", end='')
# for i in range(len(real_tokens)):
#     print(colored(f'{i:^7}', 'white'), end='')
# print()

# # Разделитель
# print("           " + "─"*7 * len(real_tokens))

# # Строка с токенами
# print("Токены:    ", end='')
# for i, token in enumerate(real_tokens):
#     color = colors[i % len(colors)]
#     display = token.replace('▁', '_')
#     print(colored(f'{display:^7}', color), end='')
# print()

# # Строка с ID токенов
# print("ID:        ", end='')
# for i, token_id in enumerate(real_ids):
#     color = colors[i % len(colors)]
#     print(colored(f'{token_id:^7}', color), end='')
# print()

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

{'input_ids': tensor([[   101,  15393,  26856,    877,  13008,    106,  89537, 105733,   1469,
          88626,   1516,  81642,    201,    102,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              

### Загрузка данных

In [5]:
df = pd.read_csv("../data/ru-hard-detection-clean-v2.csv")

df["label"] = df["source"].map({
    "human": 0,
    "ai": 1
})
df['stratify_col'] = df['label'].astype(str) + "_" + df['dataset'].astype(str)
df['group_id'] = df['dataset'] + "_" + df['id'].astype(str)
df.iloc[478:482]

,id,text,model,source,dataset,noun_ratio,verb_ratio,adj_ratio,avg_word_len,pron_ratio,perf_verb_ratio,punct_ratio,avg_syntax_links,label,stratify_col,group_id
478,479,Творчество Льва Абрамовича Кассиля : нравствен...,deepseek-chat,ai,Corus Essays,0.280702,0.096491,0.142544,6.421875,0.026316,0.318182,0.157895,1.0,1,1_Corus Essays,Corus Essays_479
479,480,"предубеждениями, доказывая, что в борьбе за Ро...",gemini-2.0-flash,ai,Corus Essays,0.325260,0.089965,0.121107,6.122449,0.031142,0.423077,0.152249,1.0,1,1_Corus Essays,Corus Essays_480
480,1,Период с 1922 по 1939 год — время становления ...,NaN,human,Corus Essays,0.355408,0.055188,0.145695,6.391645,0.013245,0.800000,0.154525,1.0,0,0_Corus Essays,Corus Essays_1
481,2,"Время, в которое развернулась деятельность зна...",NaN,human,Corus Essays,0.320713,0.077951,0.129176,6.859375,0.026726,0.657143,0.144766,1.0,0,0_Corus Essays,Corus Essays_2


### Train / test split

In [6]:
def split_data(X, y, group_by):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=group_by))

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    return X_train, X_test, y_train, y_test

In [7]:
X = df['text']
y = df['label']

X_train_full, X_test, y_train_full, y_test = split_data(X, y, df["group_id"])
train_groups = df.loc[X_train_full.index, "group_id"]
X_train, X_val, y_train, y_val = split_data(X_train_full, y_train_full, train_groups)

### Создаем датасет

In [8]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):        
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        if isinstance(idx, (list, np.ndarray)):
            idx = idx[0]

        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

### Создаем даталоадеры

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = TextDataset(
    X_train,
    y_train,
    tokenizer,
    MAX_LENGTH
)

val_dataset = TextDataset(
    X_val,
    y_val,
    tokenizer,
    MAX_LENGTH
)

test_dataset = TextDataset(
    X_test,
    y_test,
    tokenizer,
    MAX_LENGTH
)

In [10]:
train_dataset[200]

{'input_ids': tensor([   101,  58232,    128,  11267, 108893,   3788,  59421,    851,  28176,
          70227,  47721,    132,  13079,  27289,    128,  22927,  10531,  32731,
           3187,  34494,    612,  19506,    861,  82013,    876,    851,  39270,
           1641,  24863,    128,   7788,  62728,  14338,   1758,  20201,    851,
          43917,    132,  11601,   7363,    128,   2752,  37532,   1699,  53345,
           4564,  51946,    128,   3435,  17171,   1758,  36998,    845,  39798,
            851,  41530,  28377,    132,    781,  24317,   7021,  21326,    128,
           1997,  12432,  17872,  42213,   1699,  12926,  13712,    128,    625,
          94583,    128,  26882,  18380,    612,   7214,    128,  29022,    842,
            851,  19235,    842,  43384,   6297,  86447,   2785,  14591,   5206,
            132,  36018,    842, 110374,    626,  42497,  43287,  29167,  15880,
          34494,    845,  26878,  34555,    128,   1997,  12232,  12091,   5257,
           7065

In [11]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [12]:
batch = next(iter(train_loader))
batch

{'input_ids': tensor([[  101, 89663, 16518,  ...,     0,     0,     0],
         [  101,   811, 25291,  ...,     0,     0,     0],
         [  101, 15675, 11894,  ...,     0,     0,     0],
         ...,
         [  101,   811,   132,  ...,     0,     0,     0],
         [  101, 34165,   869,  ...,     0,     0,     0],
         [  101, 46157, 34867,  ...,     0,     0,     0]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]]),
 'labels': tensor([0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1])}

In [13]:
batch["input_ids"].shape

torch.Size([16, 512])

In [14]:
batch["attention_mask"].shape

torch.Size([16, 512])

In [15]:
batch["labels"].shape

torch.Size([16])

In [16]:
len(train_loader)

115

### Настройка параметров обучения

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [18]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    output_attentions=False,
    output_hidden_states=False
).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those par

#### Настройка оптимизатора

In [19]:
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)

total_steps = len(train_loader) * NUM_EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

#### Настройка метрик

In [20]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='binary')
    precision = precision_score(labels, predictions, average='binary')
    recall = recall_score(labels, predictions, average='binary')
    
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [21]:
training_args = TrainingArguments(
    output_dir="./models/bert",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, 
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=50,
    weight_decay=0.01,               # регуляризация
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    load_best_model_at_end=True,     # загрузить лучшую модель по итогам валидации
    metric_for_best_model="f1",      # ориентируемся на F1-меру    
    seed=RANDOM_SEED,
    report_to="none"
)

In [22]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

### Запуск обучения

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("./models/bert/bert_v1")
tokenizer.save_pretrained("./models/bert/bert_v1")

### Финальный тест

In [ ]:
test_results = trainer.evaluate(test_dataset)
for key, value in test_results.items():
    print(f"{key}: {value:.4f}")

#### Loss History

In [ ]:
history = trainer.state.log_history

train_loss = [x['loss'] for x in history if 'loss' in x]
eval_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]
epochs = range(1, len(eval_loss) + 1)

plt.figure(figsize=(10, 6))
plt.plot(epochs, eval_loss, 'b-o', label='Validation Loss')

plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

#### Матрица ошибок

In [ ]:
predictions = trainer.predict(test_dataset)
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=-1)
y_score = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=-1)[:, 1].numpy()

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Human', 'AI'], yticklabels=['Human', 'AI'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

#### ROC-кривая

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()